# 02_01 Bag of words: counting as a way of seeing

A model can only do arithmetic. This notebook turns text into the simplest numbers there are, counts, and
then measures what that costs: how many columns, how many of them are zero, and what the counts cannot
tell apart.

**How this notebook works.** The same rhythm as Lab 01:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-02-turning-words-into-numbers", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import csv
import json
import math
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nlpcheck import ask, guess, reveal, check_02_01

reviews3 = ["This movie is very scary and long",
            "This movie is not scary and is slow",
            "This movie is spooky and good"]
kw = list(csv.DictReader(open("data/kittiwake_reviews.csv")))
print(len(kw), "Kittiwake app reviews;", kw[0]["stars"], "stars:", kw[0]["text"])
os.makedirs("out", exist_ok=True)

## 1. Recall, from Lab 01

**r1.** Why did Lab 01's cleaning step turn "not working" into "working"?
(a) the tokenizer split it, (b) "not" is on NLTK's stop list, (c) the lemmatizer changed it

**r2.** Which reduces a word by looking it up rather than by cutting suffixes?
(a) a lemmatizer, (b) a stemmer

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. Three reviews, by hand

The chapter built a vocabulary of eleven words from the three horror-movie reviews and counted each one.
Here is the same thing in plain Python, so there is nothing hidden.

In [ ]:
vocab = []
for r in reviews3:
    for w in r.lower().split():
        if w not in vocab:
            vocab.append(w)
print(len(vocab), vocab)

for r in reviews3:
    words = r.lower().split()
    print([words.count(v) for v in vocab], "<-", r)

Each review is now a row of eleven numbers. Review 2 has a `2` under `is`, because it says "is" twice.
That grid, one row per document and one column per word, is a **document-term matrix**.

scikit-learn's `CountVectorizer` does the same in two lines. `fit` learns the vocabulary; `transform`
counts. It sorts the vocabulary alphabetically, so the columns come out in a different order:

In [ ]:
cv = CountVectorizer()
X = cv.fit_transform(reviews3)
pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=["review 1", "review 2", "review 3"])

## 3. What the bag forgets

Predict: will "dog bites man" and "man bites dog" get the same row of counts? (`True` or `False`)

In [ ]:
guess("dog_bites_man_same", None)

In [ ]:
pair = ["dog bites man", "man bites dog"]
d = CountVectorizer().fit(pair)
rows = d.transform(pair).toarray()
print(d.get_feature_names_out())
print(rows)
reveal("dog_bites_man_same", bool((rows[0] == rows[1]).all()))

Identical. The bag keeps which words and how many, and nothing about their order, so the headline and
the non-event are the same vector. **Bigrams**, pairs of neighbouring words, put some order back:

In [ ]:
d2 = CountVectorizer(ngram_range=(1, 2)).fit(pair)
print(d2.get_feature_names_out())
print(d2.transform(pair).toarray())

Now `dog bites` and `man bites` are separate columns and the rows differ. The price is more columns:
three words became seven. Keep that price in mind for section 5.

One more thing the default forgets. Predict which words of "I am a fan of it" make it into the
vocabulary.

In [ ]:
guess("fan_vocab", None)   # the words you expect, as one string, like "am fan it of" 

In [ ]:
v = CountVectorizer().fit(["I am a fan of it"]).get_feature_names_out()
reveal("fan_vocab", " ".join(v))
print("the pattern it uses:", CountVectorizer().token_pattern)

`I` and `a` are gone. `CountVectorizer`'s default token pattern, `(?u)\b\w\w+\b`, only keeps runs of
two or more letters or digits, so every one-character word disappears before counting starts. That is a
reasonable default for topic words and a problem for anything where "I" matters. Pass
`token_pattern=r"(?u)\b\w+\b"` to keep them.

## 4. Sparsity, measured on Kittiwake's reviews

A document uses a handful of words; the vocabulary holds every word any document used. So most of each
row is zero. Predict: what percentage of the entries in the Kittiwake reviews' document-term matrix are
zero? (a number from 0 to 100)

In [ ]:
guess("kw_zeros_pct", None)

In [ ]:
texts = [r["text"] for r in kw]
M = CountVectorizer().fit_transform(texts)
zeros = 100 * (1 - M.nnz / (M.shape[0] * M.shape[1]))
print("shape:", M.shape, "| non-zero entries:", M.nnz)
reveal("kw_zeros_pct", round(zeros))
print(f"{zeros:.2f} percent zeros")

M2 = CountVectorizer(ngram_range=(1, 2)).fit_transform(texts)
print("with bigrams:", M2.shape, f"{100 * (1 - M2.nnz / (M2.shape[0] * M2.shape[1])):.2f} percent zeros")

Ninety-two percent of the grid is zeros, and adding bigrams makes it ninety-six. `M` is a **sparse
matrix**: scikit-learn stores only the non-zero entries and their positions, which is why `M.nnz` (number
of non-zeros) exists. Kittiwake's reviews are short and repetitive, so this is the gentle case.

## 5. Your turn: real text

The **SMS Spam Collection** is 5,574 real text messages, published for research by the UCI Machine
Learning Repository under CC BY 4.0 (https://archive.ics.uci.edu/dataset/228/sms+spam+collection). It is
already in `data/sms_spam/`, so there is nothing to download. Lab 03 trains a spam filter on it. Real messages are messier than the invented reviews. Do exactly what section 4 did, on the
SMS messages:

- `vocab_unigram`: how many columns `CountVectorizer()` makes,
- `vocab_bigram`: how many columns `CountVectorizer(ngram_range=(1, 2))` makes,
- `zeros_unigram`, `zeros_bigram`: the percentage of zeros in each matrix.

In [ ]:
sms = [line.rstrip("\n").split("\t", 1)[1] for line in open("data/sms_spam/SMSSpamCollection", encoding="utf-8")]
print(len(sms), "messages, for example:", sms[2][:80])

vocab_unigram = None   # YOUR CODE HERE
vocab_bigram = None    # YOUR CODE HERE
zeros_unigram = None   # YOUR CODE HERE
zeros_bigram = None    # YOUR CODE HERE

In [ ]:
json.dump({"vocab_unigram": vocab_unigram, "vocab_bigram": vocab_bigram,
           "zeros_unigram": zeros_unigram, "zeros_bigram": zeros_bigram},
          open("out/02_01_bow.json", "w"), indent=1)
check_02_01();

Real text keeps inventing words: typos, names, abbreviations, numbers. The vocabulary never stops
growing, which you can watch happen:

In [ ]:
import re
seen = set()
for i, t in enumerate(sms, 1):
    seen |= set(re.findall(r"(?u)\b\w\w+\b", t.lower()))
    if i in (100, 500, 1000, 2000, 4000, len(sms)):
        print(f"after {i:5} messages: {len(seen):5} distinct words")

## 6. Exit ticket

**x1.** What does it mean that a document-term matrix is sparse? (a) it has few rows, (b) almost all of its
entries are zero, (c) it has been compressed

Explain it back: name two things a bag of words cannot tell apart, and one thing bigrams fix.

*Your explanation:* 

In [ ]:
ask("x1", "")